Prywes, N., Phillips, N.R., Oltrogge, L.M. et al. A map of the rubisco biochemical landscape. Nature 638, 823–828 (2025). https://doi.org/10.1038/s41586-024-08455-0
从中获得了Rubisco酶(EC 4)的突变扫描活性数据；可见/share/home/wangtb/olig_activity/data/activity/2024_01_30_bootstrap_Kms_Vmax.csv和/share/home/wangtb/olig_activity/data/pdb/9RUB.cif

In [3]:
import pandas as pd
import re
import os
import warnings
from Bio.PDB import PDBParser, MMCIFParser
from Bio import BiopythonWarning

warnings.simplefilter('ignore', BiopythonWarning)

DATA_DIR = '/share/home/wangtb/olig_activity/data'
PDB_DIR = os.path.join(DATA_DIR, 'pdb')
INPUT_CSV = os.path.join(DATA_DIR, 'activity_datasets_with_annotations.csv') 
REF_CSV = '/share/home/wangtb/olig_activity/raw_data/ProteinGym/reference_files/DMS_substitutions.csv'
OUTPUT_CSV = os.path.join(DATA_DIR, 'final_filtered_unique_enzymes.csv')

def get_residues_from_structure(filepath):
    if not os.path.exists(filepath):
        return set()

    if filepath.endswith('.cif'):
        parser = MMCIFParser()
    else:
        parser = PDBParser()
        
    try:
        structure = parser.get_structure('struct', filepath)
        residues = set()
        for model in structure:
            for chain in model:
                for residue in chain:
                    resseq = residue.get_id()[1]
                    residues.add(resseq)
        return residues
    except Exception as e:
        print(f"解析结构文件出错 {filepath}: {e}")
        return set()

def extract_annotation_positions(annotation_str):
    if pd.isna(annotation_str) or annotation_str == '[]':
        return []
    matches = re.findall(r'\((\d+)-\d+\)', str(annotation_str))
    return [int(m) for m in matches]

def is_consistent_major_class(ec_str):
    if pd.isna(ec_str) or not str(ec_str).strip():
        return False
    
    ecs = str(ec_str).split(';')
    major_classes = set()
    
    for ec in ecs:
        ec = ec.strip()
        if not ec: 
            continue
        major_class = ec.split('.')[0]
        if major_class.isdigit():
            major_classes.add(major_class)

    return len(major_classes) == 1

def main():
    print("1. 加载数据并进行初步筛选...")
    df = pd.read_csv(INPUT_CSV)
    
    # 筛选 1: EC 号不为空，且所有包含的 EC 号必须属于同一个大类
    mask_consistent_ec = df['ec_number'].apply(is_consistent_major_class)
    df = df[mask_consistent_ec]
    
    # 筛选 2: 非冗余去重 (对同一 UniProt_ID，只保留第一个出现的数据集)
    df = df.sort_values('DMS_id').drop_duplicates(subset=['UniProt_ID'], keep='first')
    
    # 加载 Reference CSV 获取 PDB 文件名
    ref_df = pd.read_csv(REF_CSV)[['DMS_id', 'pdb_file']]
    df = pd.merge(df, ref_df, on='DMS_id', how='left')
    
    print(f"初步筛选后剩余 {len(df)} 个大类一致的非冗余酶。开始进行 3D 结构残基验证...")
    
    valid_rows = []
    # 筛选 3: 循环遍历验证位点信息是否在 PDB 结构中存在
    for _, row in df.iterrows():
        pdb_file = row['pdb_file']
        if pd.isna(pdb_file):
            continue
            
        pdb_path = os.path.join(PDB_DIR, pdb_file)
        # 提取标注的位点编号
        positions = extract_annotation_positions(row['annotations'])
        
        if not positions:
            continue
            
        # 获取 PDB 实际含有的残基序列号
        struct_residues = get_residues_from_structure(pdb_path)
        
        # 检查是否所有提取出的位点都存在于真实结构中
        if struct_residues and all(pos in struct_residues for pos in positions):
            valid_rows.append(row)
        else:
            missing = [pos for pos in positions if pos not in struct_residues]
            print(f"[-] 剔除 {row['DMS_id']}: 标注位点 {missing} 在结构 {pdb_file} 中不存在 (可能存在编号偏移或缺失)")

    final_df = pd.DataFrame(valid_rows)
    print(f"结构验证通过: {len(final_df)} 个。")
    
    # 4. 手动整合新文献中挖掘的 Rubisco (EC 4) 数据
    rubisco_data = {
        'DMS_id': 'Rubisco_Prywes_2024',
        'UniProt_ID': 'P0A3D3', # Rhodospirillum rubrum Rubisco 的 UniProt ID
        'DMS_filename': '2024_01_30_bootstrap_Kms_Vmax.csv',
        'selection_type': 'In vivo Vmax/Kc screening',
        # 根据前文分析，填入核心催化位点
        'annotations': "['Active site(166-166): Proton transfer', 'Active site(191-191): Mg2+ coordination', 'Active site(329-329): Catalytic']",
        'ec_number': '4.1.1.39',
        'pdb_file': '9RUB.cif'
    }
    
    # 对 Rubisco 同样进行严格的结构位置验证
    rubisco_cif_path = os.path.join(PDB_DIR, rubisco_data['pdb_file'])
    rubisco_res = get_residues_from_structure(rubisco_cif_path)
    rub_pos = extract_annotation_positions(rubisco_data['annotations'])
    
    if rubisco_res and all(p in rubisco_res for p in rub_pos):
        print(f"[+] 成功: Rubisco 催化位点与 9RUB.cif 结构完美对齐。")
        final_df = pd.concat([final_df, pd.DataFrame([rubisco_data])], ignore_index=True)
    else:
        print(f"[!] 警告: Rubisco 的 9RUB.cif 结构文件未找到或位点对齐失败，但仍将被添加。")
        final_df = pd.concat([final_df, pd.DataFrame([rubisco_data])], ignore_index=True)
    
    # 清理并整理列顺序输出
    final_df = final_df[['DMS_id', 'UniProt_ID', 'DMS_filename', 'selection_type', 'annotations', 'ec_number', 'pdb_file']]
    final_df.to_csv(OUTPUT_CSV, index=False)
    
    print("\n" + "="*40)
    print(f"清洗与整合完毕！最终获得 {len(final_df)} 个高质量的单一酶类数据集。")
    print(f"数据已保存至: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

1. 加载数据并进行初步筛选...
初步筛选后剩余 18 个大类一致的非冗余酶。开始进行 3D 结构残基验证...
[-] 剔除 ENVZ_ECOLI_Ghose_2023: 标注位点 [243, 347, 373, 392, 402] 在结构 ENVZ_ECOLI.pdb 中不存在 (可能存在编号偏移或缺失)
[-] 剔除 MET_HUMAN_Estevam_2023: 标注位点 [1204, 1084, 1110] 在结构 MET_HUMAN.pdb 中不存在 (可能存在编号偏移或缺失)
[-] 剔除 PHOT_CHLRE_Chen_2023: 标注位点 [529, 120, 410, 433] 在结构 PHOT_CHLRE.pdb 中不存在 (可能存在编号偏移或缺失)
结构验证通过: 13 个。
[+] 成功: Rubisco 催化位点与 9RUB.cif 结构完美对齐。

清洗与整合完毕！最终获得 14 个高质量的单一酶类数据集。
数据已保存至: /share/home/wangtb/olig_activity/data/final_filtered_unique_enzymes.csv


In [4]:
import pandas as pd
import re
import os
import requests
import warnings
from Bio.PDB import PDBParser, MMCIFParser
from Bio.PDB.Polypeptide import protein_letters_3to1
from Bio import BiopythonWarning

# 忽略结构解析警告
warnings.simplefilter('ignore', BiopythonWarning)

# --- 路径配置 ---
DATA_DIR = '/share/home/wangtb/olig_activity/data'
PDB_DIR = os.path.join(DATA_DIR, 'pdb')
INPUT_CSV = os.path.join(DATA_DIR, 'final_filtered_unique_enzymes.csv')
OUTPUT_REPORT = os.path.join(DATA_DIR, 'residue_validation_report.csv')

def get_uniprot_sequence(uniprot_id):
    """通过 UniProt API 获取蛋白质全长 FASTA 序列"""
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            # 去除第一行 (Fasta Header)，拼接序列
            lines = response.text.strip().split('\n')
            seq = ''.join(lines[1:])
            return seq
    except Exception as e:
        print(f"获取 {uniprot_id} 序列失败: {e}")
    return ""

def safe_3to1(resname):
    """安全的 3字母到1字母氨基酸转换，兼容常见修饰残基"""
    mapping = protein_letters_3to1.copy()
    # 添加常见的酶活性中心修饰氨基酸
    mapping['KCX'] = 'K' # 氨基甲酰化赖氨酸 (如 Rubisco K191)
    mapping['CSO'] = 'C' # 羟基化半胱氨酸
    mapping['PTR'] = 'Y' # 磷酸化酪氨酸
    mapping['SEP'] = 'S' # 磷酸化丝氨酸
    mapping['TPO'] = 'T' # 磷酸化苏氨酸
    mapping['SEC'] = 'U' # 硒代半胱氨酸
    
    return mapping.get(resname.upper(), '?')

def get_pdb_residues(filepath, positions):
    """解析 PDB 文件，返回指定位置的1字母氨基酸代码"""
    if not os.path.exists(filepath):
        return {}
    
    parser = MMCIFParser() if filepath.endswith('.cif') else PDBParser()
        
    try:
        structure = parser.get_structure('struct', filepath)
        res_dict = {}
        # 遍历模型和链，寻找对应编号的残基
        for model in structure:
            for chain in model:
                for residue in chain:
                    resseq = residue.get_id()[1]
                    if resseq in positions:
                        resname = residue.get_resname()
                        # 如果同一个位置由于多链有多个残基，我们取第一个匹配的即可（默认同源多聚体序列一致）
                        if resseq not in res_dict:
                            res_dict[resseq] = safe_3to1(resname)
        return res_dict
    except Exception as e:
        print(f"解析 {filepath} 失败: {e}")
        return {}

def extract_annotation_positions(annotation_str):
    """提取注释字符串中的残基编号"""
    if pd.isna(annotation_str) or annotation_str == '[]':
        return []
    matches = re.findall(r'\((\d+)-\d+\)', str(annotation_str))
    return [int(m) for m in matches]

def main():
    print(f"开始校验序列与结构的残基一致性...\n读取文件: {INPUT_CSV}")
    df = pd.read_csv(INPUT_CSV)
    
    report_rows = []
    perfect_match_count = 0

    for _, row in df.iterrows():
        dms_id = row['DMS_id']
        uniprot_id = row['UniProt_ID']
        pdb_file = row['pdb_file']
        
        positions = extract_annotation_positions(row['annotations'])
        if not positions:
            continue
            
        print(f"[{dms_id}] 正在获取 UniProt 序列...")
        seq = get_uniprot_sequence(uniprot_id)
        if not seq:
            print(f"  -> 无法获取序列，跳过。")
            continue
            
        pdb_path = os.path.join(PDB_DIR, pdb_file)
        pdb_res_dict = get_pdb_residues(pdb_path, positions)
        
        all_match = True
        details = []

        for pos in positions:
            if pos - 1 < len(seq):
                uniprot_aa = seq[pos - 1]
            else:
                uniprot_aa = 'OUT_OF_BOUNDS'
                
            pdb_aa = pdb_res_dict.get(pos, 'MISSING')
            
            is_match = (uniprot_aa == pdb_aa)
            if not is_match:
                all_match = False
                
            details.append(f"{pos}:{uniprot_aa}(UniProt)-{pdb_aa}(PDB)")
            
        if all_match:
            perfect_match_count += 1
            print(f"  -> [成功] 位点比对完全一致: {', '.join(details)}")
        else:
            print(f"  -> [不匹配] 存在差异: {', '.join(details)}")
            
        report_rows.append({
            'DMS_id': dms_id,
            'UniProt_ID': uniprot_id,
            'Positions': str(positions),
            'Match_Status': 'Perfect' if all_match else 'Mismatch/Missing',
            'Details': " | ".join(details)
        })

    report_df = pd.DataFrame(report_rows)
    report_df.to_csv(OUTPUT_REPORT, index=False)
    
    print("\n" + "="*40)
    print(f"校验完成！")
    print(f"总计验证了 {len(report_rows)} 个蛋白质的活性位点。")
    print(f"准确无误 (UniProt 序列与 PDB 结构残基 100% 对应) 的蛋白数量: {perfect_match_count}")
    print(f"详细校验报告已保存至: {OUTPUT_REPORT}")
    print("注：如果不匹配，通常是因为表达载体带有 Tag、截短表达、或实验使用了不同物种的同源结构。")

if __name__ == "__main__":
    main()

开始校验序列与结构的残基一致性...
读取文件: /share/home/wangtb/olig_activity/data/final_filtered_unique_enzymes.csv
[AICDA_HUMAN_Gajula_2014_3cycles] 正在获取 UniProt 序列...
  -> [成功] 位点比对完全一致: 58:E(UniProt)-E(PDB), 56:H(UniProt)-H(PDB), 87:C(UniProt)-C(PDB), 90:C(UniProt)-C(PDB)
[AMIE_PSEAE_Wrenbeck_2017] 正在获取 UniProt 序列...
  -> [成功] 位点比对完全一致: 59:E(UniProt)-E(PDB), 134:K(UniProt)-K(PDB), 166:C(UniProt)-C(PDB)
[CAS9_STRP1_Spencer_2017_positive] 正在获取 UniProt 序列...
  -> [成功] 位点比对完全一致: 10:D(UniProt)-D(PDB), 840:H(UniProt)-H(PDB), 10:D(UniProt)-D(PDB), 10:D(UniProt)-D(PDB), 762:E(UniProt)-E(PDB), 766:E(UniProt)-E(PDB), 766:E(UniProt)-E(PDB), 983:H(UniProt)-H(PDB), 1297:H(UniProt)-H(PDB), 1328:D(UniProt)-D(PDB)
[CASP3_HUMAN_Roychowdhury_2020] 正在获取 UniProt 序列...
  -> [不匹配] 存在差异: 121:H(UniProt)-C(PDB), 163:C(UniProt)-E(PDB)
[CASP7_HUMAN_Roychowdhury_2020] 正在获取 UniProt 序列...
  -> [不匹配] 存在差异: 144:H(UniProt)-F(PDB), 186:C(UniProt)-N(PDB)
[LGK_LIPST_Klesmith_2015] 正在获取 UniProt 序列...
  -> [成功] 位点比对完全一致: 23:T(UniProt)-T(P

In [5]:
import pandas as pd
import re
import os

DATA_DIR = '/share/home/wangtb/olig_activity/data'
INPUT_ENZYMES = os.path.join(DATA_DIR, 'final_filtered_unique_enzymes.csv')
INPUT_REPORT = os.path.join(DATA_DIR, 'residue_validation_report.csv')
OUTPUT_DATASET = os.path.join(DATA_DIR, 'dataset.csv')

def parse_active_residues(details_str):
    if pd.isna(details_str):
        return ""
    
    matches = re.findall(r'(\d+):([A-Z])\(UniProt\)', details_str)
    
    formatted = [f"{aa}{pos}" for pos, aa in matches]
    return ";".join(formatted)

def main():
    print("正在生成最终数据集...")
    
    if not os.path.exists(INPUT_ENZYMES) or not os.path.exists(INPUT_REPORT):
        print("错误：找不到输入的 CSV 文件或校验报告，请确保之前的脚本已运行。")
        return

    enzymes_df = pd.read_csv(INPUT_ENZYMES)
    report_df = pd.read_csv(INPUT_REPORT)
    
    valid_ids = report_df[report_df['Match_Status'] == 'Perfect'][['DMS_id', 'Details']]
    
    df = pd.merge(enzymes_df, valid_ids, on='DMS_id', how='inner')
    
    print(f"校验通过并保留的蛋白数量: {len(df)}")
    
    df['ec_major_class'] = df['ec_number'].apply(lambda x: str(x).split('.')[0] if pd.notna(x) else "")
    

    df['active_residues'] = df['Details'].apply(parse_active_residues)
    
    output_df = df[[
        'DMS_id', 
        'UniProt_ID', 
        'DMS_filename', 
        'pdb_file',
        'ec_major_class', 
        'active_residues', 
        'selection_type'
    ]].copy()
    
    output_df.columns = ['DMS_id', 'UniProt_ID', 'DMS_filename', 'pdb_file', 'EC_Major', 'Active_Sites', 'Selection_Type']
    
    output_df.to_csv(OUTPUT_DATASET, index=False)
    
    print("\n" + "="*40)
    print(f"成功生成数据集！")
    print(f"文件路径: {OUTPUT_DATASET}")
    print(f"总计条目: {len(output_df)}")
    print("\n数据预览 (前 5 行):")
    print(output_df.head())

if __name__ == "__main__":
    main()

正在生成最终数据集...
校验通过并保留的蛋白数量: 10

成功生成数据集！
文件路径: /share/home/wangtb/olig_activity/data/dataset.csv
总计条目: 10

数据预览 (前 5 行):
                             DMS_id   UniProt_ID  \
0   AICDA_HUMAN_Gajula_2014_3cycles  AICDA_HUMAN   
1          AMIE_PSEAE_Wrenbeck_2017   AMIE_PSEAE   
2  CAS9_STRP1_Spencer_2017_positive   CAS9_STRP1   
3           LGK_LIPST_Klesmith_2015    LGK_LIPST   
4                 OTC_HUMAN_Lo_2023    OTC_HUMAN   

                           DMS_filename         pdb_file EC_Major  \
0   AICDA_HUMAN_Gajula_2014_3cycles.csv  AICDA_HUMAN.pdb        3   
1          AMIE_PSEAE_Wrenbeck_2017.csv   AMIE_PSEAE.pdb        3   
2  CAS9_STRP1_Spencer_2017_positive.csv   CAS9_STRP1.pdb        3   
3           LGK_LIPST_Klesmith_2015.csv    LGK_LIPST.pdb        2   
4                 OTC_HUMAN_Lo_2023.csv    OTC_HUMAN.pdb        2   

                                        Active_Sites       Selection_Type  
0                                    E58;H56;C87;C90  bulk RNA-sequencing  


SRC_HUMAN_Ahler_2019,SRC_HUMAN,SRC_HUMAN_Ahler_2019.csv,SRC_HUMAN.pdb,2,D389;L276;K298,Growth，这个蛋白没有产生任何同源fasta，从数据集中剔除